# Summary Algorithm (Table1)
_Note that this is the same algorithm as in `3-data-preparation.ipynb` 
but without collection of the subtasks and with a different visualization_

In [ ]:
# The summary algorithm computes a lot of descriptive statistics. I suggest to have a
# brief look at the swimlane diagram:
# https://algorithms.vantage6.ai/en/latest/v6-summary-py/docs/v6-summary-py/implementation.html#overview
# to have a good overview of the different steps in the algorithm. From the diagram
# you can see that this is a two (federated-)step algorithm:
#
# 1. Call `summary_per_data_station`
# 2. Call `variance_per_data_station`
#
# And then there is the central part responsible for the aggregation of the results. The
# central part of the algorithm (the main call) will return the summary statistics for
# the entire federated dataset. In IDEA4RC, the summary statistics per data station are
# also required. So in this notebook we go through the following steps to obtain both
# the *global* (from the central part) and the *local* (from the
# `summary_per_data_station` call) summary statistics:
#
# 1. Create a new vantage6 task to execute the *summary* method (central part). This
#    central part will start the tasks `summary_per_data_station` and
#    `variance_per_data_station` (as you can see in the swimlane diagram).
# 2. Poll until the task is finished
# 3. Retrieve the *global* summary statistics from the central part (the main call)
# 4. Retrieve the *local* summary statistics from the data stations (the
#    `summary_per_data_station` call that was made by the central part)
#


In [ ]:
import base64
import json
import requests

In [ ]:
with open("token.txt", "r") as f:
    token = f.read().strip()
headers = {
    "Authorization": token
}

In [ ]:
# Set the collaboration ID:
#
#   - 2: Test collaboration (with IKNL and UPM)
#   - 3: IDEA4RC collaboration
#
COLLABORATION_ID = 2

In [ ]:
# Set the organization IDs that are part of this workspace. These should be the IDs of
# the vantage6 organizations:
#
#   1	- root
#   2	- ENG
#   3	- UPM
#   4	- INT
#   5	- UKE
#   6	- CLB
#   7	- FPNS
#
# These are basically all organization that are part of the workspace (thus the same
# list as in the `1-new-workspace.ipynb` notebook):
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/organization?collaboration_id={COLLABORATION_ID}",
    headers=headers
)
ORGANIZATIONS = {org["id"]: org["name"] for org in response.json()["data"]}
ORGANIZATION_IDS = list(ORGANIZATIONS.keys())
ORGANIZATION_NAMES = list(ORGANIZATIONS.values())

ORGANIZATIONS

In [ ]:
# Set the study ID. This is the `study` id that belongs to the RAVEN workspace. See the
# `0-new-workspace.ipynb` notebook for more information.
STUDY_ID = 4
# Set the session ID. This is the `v6_session` id that belongs to the RAVEN analysis.
# See the `1-new-analysis.ipynb` notebook for more information.
SESSION_ID = 3

In [ ]:
# The image to use is the latest version of the analytics algorithm
IMAGE = "harbor2.vantage6.ai/idea4rc/analytics:latest"
#
# The method (that is within this IMAGE) to execute is the summary algorithm. For
# data exploration we use the `summary` method.
METHOD = "summary"

In [ ]:
# The user is able to select multiple cohorts in RAVEN. These cohorts correspond to
# different dataframes, see the `2-new-cohort.ipynb` how these are created and check
# the `v6_dataframe` column in the RAVEN database for more information.
#
# In this example I've just used one cohort. Simply add more ids to the list to use
# more cohorts.
DATAFRAME_IDS = [99, 100]

In [ ]:
# Before we can start analysis the cohorts (dataframes) we need to check the variables
# that are available in the dataframes. You can use this endpoint whenever you want user
# to allow you to select variables.
# TODO the dtpyes might change in the future, so do not rely on them to heavily now. In
# the next version of the data extraction job we will likely provide you with either the
# `category` or `numeric` colum type (so that you can use them to select varables)
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/session/dataframe/{DATAFRAME_IDS[0]}",
    headers=headers
)
VARIABLES = response.json()["columns"]
VARIABLES

In [ ]:
org_input = [
    {
        "id": ORGANIZATION_IDS[0], # Central task
        "arguments": base64.b64encode(
            json.dumps(
                {
                    # "columns": VARIABLES,
                    # "numeric_columns": NUMERIC_VARIABLES,
                    "organizations_to_include": ORGANIZATION_IDS # all participants
                }
            ).encode("UTF-8")
        ).decode("UTF-8")
    }
]

payload = {
    "name": "Human-readable name of the task",
    "image": IMAGE,
    "description": "Description of the task",
    "action": "central_compute",
    "method": METHOD,
    "organizations": org_input,
    "databases": [
        [
            {
                "type": "dataframe",
                "dataframe_id": df_id
            } for df_id in DATAFRAME_IDS
        ]
    ],
    "session_id": SESSION_ID,
    "study_id": STUDY_ID
}
payload

In [ ]:
# Then using the authorization header and the payload we can create a vantage6 task
# using the vantage6 server API.
response = requests.post(
    "https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/task",
    headers=headers,
    json=payload
)
# In the response we need to extract the task ID and the job ID so we can poll werther
# the (central) task is finished. Later on we can also use these IDs to retrieve the
# results.
TASK_ID = response.json()["id"]
JOB_ID = response.json()["job_id"]
response.json()

In [ ]:
# Poll until the (central) task is finished. We do not concern about the subtasks in
# this instance. We could consider including them in the future as well?
response = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/run?task_id={TASK_ID}",
    headers=headers,
)
# Since it is a central task we can obtain the [0]th element of the data list as it
# always should be a single element in a list. Wait until the status returns
# "completed".
response.json()["data"][0]["status"]

In [ ]:
# Get the results of the (central) task, thus the *global* summary statistics.
response_global = requests.get(
    f"https://vantage6-core.orchestrator.idea.lst.tfo.upm.es/server/result?task_id={TASK_ID}",
    headers=headers,
)
# Again, since it is a central task we can obtain the [0]th element of the data list
result_global = [json.loads(base64.b64decode(result["result"]).decode("UTF-8")) for result in response_global.json()["data"]]
result_global

# Visualization of results
# Table1

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def fmt_mean_sd(stats):
    if np.isnan(stats['mean']):
        return "NaN (NaN)"
    return f"{round(stats['mean'], 1)} ({round(stats['std'], 1)})"

def fmt_int(value):
    if np.isnan(value):
        return "NaN"
    return f"{int(value)}"

def get_numeric_rows(result, cohorts, var):
    """Return numeric summary rows (mean, min, max)."""
    def numeric(cohort):
        return result[cohort]['numeric'][var]

    return [
        ("Mean (std)", True, 
         *[fmt_mean_sd(numeric(c)) for c in cohorts]),
        ("Min", True, 
         *[fmt_int(numeric(c)['min']) for c in cohorts]),
        ("Max", True, 
         *[fmt_int(numeric(c)['max']) for c in cohorts]),
        ("Missing", True, 
        *[fmt_int(numeric(c)['missing']) for c in cohorts]),
    ]

def get_categorical_rows(result, cohort_names, var):
    """
    Return categorical summary rows with counts and percentages,
    plus a 'Missing' row based on any 'N/A' keys.
    """
    def val(cohort, key):
        return result[cohort]['counts_unique_values'][var].get(key, 0)

    def format_val(count, total):
        pct = (count / total * 100) if total > 0 else 0
        return f"{count} ({pct:.1f}%)"

    # Calculate total per cohort (including N/A)
    totals = []
    for cohort_name in cohort_names:
        total = sum(result[cohort_name]['counts_unique_values'][var].values())
        totals.append(total)

    rows = []

    # flatten nested list of keys and get unique values
    keys_nested = [list(result[cohort_name]['counts_unique_values'][var].keys()) for cohort_name in cohort_names]
    keys_flat = list(set(k for sub in keys_nested for k in sub))
    keys_flat.sort()

    # missing values
    missing_values = ['N/A', 'N/A2']

    # Normal keys
    for key_ in keys_flat:
        if not key_ in missing_values:        
            rows.append((key_.capitalize(), True, 
                         *[format_val(val(cohort_name, key_), totals[i]) for i, cohort_name in enumerate(cohort_names)],))

    # Missing keys
    missing_keys = []
    for cohort_name in cohort_names:
        values = result.get(cohort_name, {}).get('counts_unique_values', {}).get(var, {})
        for k in values.keys():
            if isinstance(k, str) and (k in missing_values):
                missing_keys.append(k)

    if missing_keys:
        missing_counts = [sum(val(cohort_name, k) for k in missing_keys) for cohort_name in cohort_names]
    else:
        missing_counts = [0, 0, 0]

    rows.append(("Missing", True, 
                 *[format_val(missing_counts[i], totals[i]) for i in range(len(cohort_names))],))

    return rows

result = result_global[0]

cohort_names = list(result.keys())
n_cohorts = len(cohort_names)

vars_num = list(result[cohort_names[0]]['numeric'].keys())
vars_cat = list(result[cohort_names[0]]['counts_unique_values'].keys())

data = []

# Numeric variables
for var_num in vars_num:
    ar = (var_num.capitalize().replace('_', ' '), False) + ("",)*(n_cohorts)
    data.append(ar)
    data += get_numeric_rows(result, cohort_names, var_num)

# Categorical examples
for var_cat in vars_cat:
    ar = (var_cat.capitalize().replace('_', ' '), False) + ("",)*(n_cohorts)
    data.append(ar)

    data += get_categorical_rows(result, cohort_names, var_cat)

# data

In [ ]:
columns_ = ["Characteristic", "is_sub"]
for cohort_name in cohort_names:
    columns_ += [f"{cohort_name} (n={result[cohort_name]['num_rows']})"] 

df_filled = pd.DataFrame(data, columns = columns_)

# Apply styling function as before
def style_rows(row):
    idx = row.name
    if not df_filled.loc[idx, "is_sub"]:
        return ["font-weight: bold;"] * len(row)
    else:
        return ["padding-left: 20px"] + [""] * (len(row) - 1)

styled_df = (
    df_filled.drop(columns=["is_sub"]).style
    .apply(style_rows, axis=1)
    .hide(axis="index")
    .set_properties(subset=["Characteristic"], **{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("font-weight", "bold"),
                                                    ("text-align", "center")]}])
)

styled_df